# 19. 통계검정과 신뢰구간 계산

실제 추론 결과의 그룹 차이가 우연인지 bootstrap confidence interval로 확인합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

In [ ]:
run_dir = paths.runs_root / "baseline_segformer_b0"
sample_metrics, group_metrics, class_metrics = load_run_metrics(run_dir)

## 19-1. 사전 정의 비교쌍

In [ ]:
tests = []
if {"red", "blue"}.issubset(set(sample_metrics["color_group"])):
    tests.append(compare_two_groups(sample_metrics, "color_group", "red", "blue", "target_dice"))
if {"scratch", "impact"}.issubset(set(sample_metrics["defect_type"])):
    tests.append(compare_two_groups(sample_metrics, "defect_type", "scratch", "impact", "target_dice"))
shapes = sorted(sample_metrics["shape_group"].unique())
if len(shapes) >= 2:
    tests.append(compare_two_groups(sample_metrics, "shape_group", shapes[0], shapes[1], "target_dice"))

test_df = pd.DataFrame(tests)
test_df.to_csv(paths.runs_root / "hypothesis_test_summary_b0.csv", index=False, encoding="utf-8-sig")
display(test_df)

## 19-2. 귀무가설 판단

In [ ]:
for _, row in test_df.iterrows():
    decision = "기각" if row["reject_h0_ci_excludes_0"] else "기각하지 않음"
    print(
        f"{row['group_col']} {row['group_a']} vs {row['group_b']}: "
        f"diff={row['diff_a_minus_b']:.3f}, 95% CI=({row['ci95_low']:.3f}, {row['ci95_high']:.3f}) -> H0 {decision}"
    )